In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Device of the full run. 'auto' (the default) takes the first available of
# 'cuda' (NVIDIA GPU), 'xla' (TPU), 'mps' (Apple GPU, only when Python runs on
# macOS itself) and 'cpu'; the cell prints the one it took. The small run
# always uses the CPU.
DEVICE = 'auto'
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}

# The device of the training: always the CPU in the small run, DEVICE otherwise.
device = src.utils.device.select_device('cpu' if IS_UNITTEST else DEVICE)
print(f"Training device: {src.utils.device.describe(device)}"
      + (" (the small run always uses the CPU)" if IS_UNITTEST else f" (DEVICE = {DEVICE!r})"))


# Initialization: Where Training Starts

Gradient descent starts from the initial weights, and in a deep network a
bad start is often fatal: the signal dies out or explodes on its way
through the layers, and so do the gradients on their way back.

Take a layer $y = W x$ with `fan_in` inputs, independent weights of mean 0
and variance $\mathrm{Var}(w)$, independent inputs. Then

$$\mathrm{Var}(y) = \text{fan\_in} \cdot \mathrm{Var}(w) \cdot \mathbb{E}[x^2].$$

A ReLU zeroes half of a symmetric signal, so after it $\mathbb{E}[x^2]$ is
half of the variance before it. Every layer multiplies the signal variance
by $\tfrac12\,\text{fan\_in} \cdot \mathrm{Var}(w)$; after 10 layers, by
its 10th power. The signal keeps its size only for

$$\mathrm{Var}(w) = \frac{2}{\text{fan\_in}} \qquad \text{(He initialization)}.$$

A uniform distribution $U(-a, a)$ has variance $a^2 / 3$, so its He version
is $a = \sqrt{6 / \text{fan\_in}}$.

**Files to edit:** [src/models/feedforward/initialization.py](../../source/code/src/models/feedforward/initialization.py).


In [ ]:
src.utils.seed.seed_all(0)


# The Data and the Network

MNIST from `MNISTSimpleDataset`, as before, and a plain network of 10
linear layers, 256 neurons wide, with ReLU between them: the
`DeepFullyConnectedNet` of the previous task with a `Linear` + `ReLU` block,
no normalization and no residual connections.


In [ ]:
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
print('training images:', len(MNIST_train), 'validation images:', len(MNIST_valid))


def make_plain_network():
    return src.models.feedforward.deep_batchnorm_net.DeepFullyConnectedNet(
        block=lambda n: torch.nn.Sequential(torch.nn.Linear(n, n), torch.nn.ReLU()),
        dim_input=28 * 28, dim_embed=256, dim_output=10, n_blocks=8)


print(make_plain_network())


# Task 1: Initialize the Weights

## Goal

Implement `initialize_(network, method, std, bound, generator)` in
[src/models/feedforward/initialization.py](../../source/code/src/models/feedforward/initialization.py). It changes every
`torch.nn.Linear` of the network in place: the bias becomes 0, the weight
$W$ (`[out_features, in_features]`, so `fan_in = in_features`) depends on
`method`:

| method | weights |
|---|---|
| `'zeros'` | $W = 0$ |
| `'normal'` | $W \sim N(0, \text{std}^2)$ |
| `'uniform'` | $W \sim U(-\text{bound}, \text{bound})$ |
| `'he_normal'` | $W \sim N(0, 2 / \text{fan\_in})$ |
| `'he_uniform'` | $W \sim U(-\sqrt{6 / \text{fan\_in}}, \sqrt{6 / \text{fan\_in}})$ |

Change the tensors in place without gradients: `with torch.no_grad():` and
`weight.normal_(0.0, std, generator=generator)`,
`weight.uniform_(-bound, bound, generator=generator)`, `bias.zero_()`.
Pass `generator` to every random call, so that the same generator gives the
same weights.


In [ ]:
initializations = {
    'zeros': dict(method='zeros'),
    'normal, std 1': dict(method='normal', std=1.0),
    'normal, std 0.01': dict(method='normal', std=0.01),
    'uniform, bound 1': dict(method='uniform', bound=1.0),
    'he_normal': dict(method='he_normal'),
    'he_uniform': dict(method='he_uniform'),
}

answer['initialization'] = {}
for name, arguments in initializations.items():
    network = make_plain_network()
    # Edit: ../../source/code/src/models/feedforward/initialization.py
    src.models.feedforward.initialization.initialize_(
        network, generator=torch.Generator().manual_seed(seed), **arguments)
    # ^^^ This function should be implemented in this task
    linears = [layer for layer in network.modules() if isinstance(layer, torch.nn.Linear)]
    answer['initialization'][name] = {
        'first_weights': linears[0].weight[:3, :5],
        'weight_stds': [layer.weight.std() for layer in linears],
        'bias_sums': [layer.bias.abs().sum() for layer in linears],
    }
    print(f"{name:18s} weight std per layer:", np.round([float(s) for s in answer['initialization'][name]['weight_stds']], 4))


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Initialize the Weights', answer,
    prefixes=('initialization',),
    is_small_run=IS_UNITTEST)


# The Signal Through the Layers

For every initialization, one batch of validation images goes through a
fresh network; the plot shows the standard deviation of the signal after
every ReLU (`activation_stds`, given). A straight line on the log scale
means the signal is multiplied by the same factor at every layer.


In [ ]:
batch_size = 32 if IS_UNITTEST else 128
train_dl = torch.utils.data.DataLoader(
    MNIST_train, batch_size=batch_size, shuffle=True, drop_last=True,
    generator=torch.Generator().manual_seed(0))
valid_dl = torch.utils.data.DataLoader(MNIST_valid, batch_size=1000, shuffle=False)
signal_batch = next(iter(valid_dl))['image'][:256]

plt.figure(figsize=(8, 4))
for name, arguments in initializations.items():
    network = make_plain_network()
    src.models.feedforward.initialization.initialize_(
        network, generator=torch.Generator().manual_seed(0), **arguments)
    stds = src.models.feedforward.initialization.activation_stds(network, signal_batch)
    stds = [std if std > 0 else float('nan') for std in stds]
    label = name + (' (all zero)' if all(math.isnan(std) for std in stds) else '')
    plt.plot(range(1, len(stds) + 1), stds, 'o-', label=label)
plt.yscale('log')
plt.xlabel('ReLU number')
plt.ylabel('standard deviation of the signal')
plt.legend()
plt.show()


# One Epoch of Training

Each network trains for one epoch with `train_model` and SGD with momentum
(`lr=0.01`). A network whose loss becomes NaN (not a number: the numbers
overflowed) has diverged.


In [ ]:
def loss(batch):
    return F.cross_entropy(batch['signals']['output'], batch['data']['label'])


def accuracy(batch):
    return (batch['postprocessed']['class'] == batch['data']['label']).float().sum(), len(batch['data']['label'])


results = {}
for name, arguments in initializations.items():
    src.utils.seed.seed_all(0)
    network = make_plain_network()
    src.models.feedforward.initialization.initialize_(
        network, generator=torch.Generator().manual_seed(0), **arguments)
    optimizer = torch.optim.SGD(network.parameters(), lr=0.01, momentum=0.9)
    history = src.train_loop.train_model(
        network, 1, train_dl, valid_dl, {'ce': loss}, optimizer,
        metrics={'accuracy': accuracy}, run_name=name,
        device=device)
    results[name] = {'valid_loss': history['valid_loss']['ce'][-1],
                     'valid_accuracy': history['valid_metrics']['accuracy'][-1]}

for name, result in results.items():
    print(f"{name:18s} validation loss {result['valid_loss']:10.4f}, accuracy {result['valid_accuracy']:.3f}")
# The grader does not accept NaN: a diverged value is saved as -1.
answer['experiments'] = {name: np.nan_to_num(list(result.values()), nan=-1.0, posinf=-1.0, neginf=-1.0)
                         for name, result in results.items()}

plt.figure(figsize=(8, 3))
plt.bar(list(results), [result['valid_accuracy'] for result in results.values()])
plt.xticks(rotation=20)
plt.ylabel('validation accuracy after 1 epoch')
plt.show()


In [ ]:
course_utils.checker.print_progress(
    'One Epoch of Training', answer,
    prefixes=('experiments',),
    is_small_run=IS_UNITTEST)


What to look for (full run, about half a minute on a laptop CPU):

- **zeros**: every signal is 0 and every neuron of a layer is the same
  function of the same inputs, so they all get the same gradient (here,
  zero for all hidden weights) and stay equal forever: the network never
  learns, accuracy about 10%, a guess;
- **normal, std 1** and **uniform, bound 1**: the signal grows about
  10 times per layer, to about $10^9$ at the end; the first steps overflow
  and the loss becomes NaN;
- **normal, std 0.01**: the signal shrinks about 10 times per layer, to
  about $10^{-9}$; the gradients of the first layers vanish with it and one
  epoch only reaches about 11%;
- **he_normal** and **he_uniform**: the signal keeps its size through all
  10 layers, and one epoch reaches 92–94%. Only the variance matters,
  not the shape of the distribution.

PyTorch's own default for `torch.nn.Linear` is uniform with bound
$1/\sqrt{\text{fan\_in}}$, a variance 6 times smaller than He's: fine
for shallow networks, too small for deep plain ones. Batch normalization
and residual connections of the previous task make deep networks far less
sensitive to the initialization.


In [ ]:
course_utils.checker.save_answer(answer)
